# ENEC 2026 · Day 3 · A Tool-Calling Agent: SQL + Documents + Forecast

**Time:** about 75 minutes  |  **Runs in:** Google Colab (free tier is fine)  |  **Needs:** OpenAI key in Colab Secrets (`OPENAI_API_KEY`)

**Data:** a small slice of the synthetic *Nuclear Enterprise 360* dataset, focused on **Asset A-001** (a cooling-water pump). All data is synthetic training data.

**Goal:** Build an agent from scratch with the OpenAI tool-calling API. It chooses between structured data (SQL), document knowledge (RAG) and analysis (forecast), with guardrails.

> Training notice: nothing in this notebook is an engineering diagnosis or a maintenance authorisation. The AI supports a qualified human reviewer.

In [ ]:
# Install what Colab does not already have (about 20 seconds)
%pip install -q -U openai 

In [ ]:
# Get the course files (data, documents, skills) and the helper module.
# In Colab this clones the public course repo into /content/enec2026oct.
import os, sys, subprocess

REPO_URL = "https://github.com/Decoding-Data-Science/enec2026oct"
IN_COLAB = os.path.isdir("/content") and "google.colab" in sys.modules
ROOT = "/content/enec2026oct" if IN_COLAB else os.getcwd()

if IN_COLAB and not os.path.isdir(ROOT):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, ROOT], capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(
            "Could not clone the course repo:\n" + r.stderr[-300:] +
            "\nFallback: upload enec2026oct.zip in the Files panel (left sidebar), "
            "run  !unzip -q enec2026oct.zip -d /content  and run this cell again."
        )

sys.path.insert(0, os.path.join(ROOT, "colab"))
import enec_colab as enec
print("Course files ready at:", enec.ROOT)

In [ ]:
# Connect to OpenAI. The key comes from Colab Secrets and is never printed.
client = enec.get_client()
MODEL = enec.pick_model(client)     # first available of: gpt-5.4-mini, gpt-5-mini, gpt-4o-mini

## 1. A question RAG alone cannot answer

> What was happening in A-001's data, what does the current approved procedure say, and what should be prepared for human reliability review?

That needs **SQL** (facts), **RAG** (documents) and **analysis** (trend). An agent decides which capability to use, in what order, and when to stop.

```
User question
    ↓
Agent (LLM + instructions)
    ↓  chooses a tool
 ┌──────┬────────────┬──────────┐
 SQL    Documents    Forecast
 └──────┴────────────┴──────────┘
    ↓  results go back to the agent
Final answer with evidence  →  human review
```

In [ ]:
import json, pandas as pd

tables = enec.load_tables()
con    = enec.make_db(tables)
docs   = enec.load_documents()
index  = enec.RAGIndex.build(client, docs)
hourly = tables["a001_sensor_hourly_90d"]
print("ready:", len(index.chunks), "chunks,", len(tables), "tables")

## 2. Tools are just functions plus a description

The model never runs code. It **asks** for a tool call as JSON; our code runs the function and returns the result. Three tools:

In [ ]:
def sql_query(query: str) -> str:
    """Tool 1: read-only SQL on the Asset 360 tables."""
    return enec.run_sql(con, query).to_csv(index=False)

def search_documents(query: str, only_approved: bool = False) -> str:
    """Tool 2: semantic search over A-001 documents. Results carry ID, status and version."""
    hits = index.search(query, k=4, statuses=["APPROVED"] if only_approved else None)
    return enec.format_hits(hits, 700)

def forecast(horizon_hours: int = 168) -> str:
    """Tool 3: the Ridge forecast (optional extension 2), summarised."""
    r = enec.forecast_vibration(hourly, int(horizon_hours)); r.pop("_forecast_df")
    return json.dumps(r)

print(sql_query("SELECT asset_id, health_score, risk_level FROM asset_360 WHERE asset_id='A-001'"))

### The tool descriptions are part of the prompt

The model decides which tool to call from these descriptions. Vague descriptions cause wrong tool choices.

In [ ]:
TOOLS = [
    enec.tool_spec("sql_query",
        "Run ONE read-only SQLite SELECT on the A-001 / Asset 360 tables. Use for counts, health scores, work orders, inspections, sensor statistics.",
        {"query": {"type": "string", "description": "A single SELECT statement."}}),
    enec.tool_spec("search_documents",
        "Semantic search over the A-001 documents (policy, procedures, reports, work order, field report). Results show document ID, approval status and version.",
        {"query": {"type": "string"}, "only_approved": {"type": "boolean", "description": "true = APPROVED documents only"}}, ["query"]),
    enec.tool_spec("forecast",
        "Statistical 7-day vibration forecast for A-001 (Ridge model from optional extension 2). Not a failure prediction.",
        {"horizon_hours": {"type": "integer", "description": "Default 168."}}, []),
]
IMPLS = {"sql_query": sql_query, "search_documents": search_documents, "forecast": forecast}
print(json.dumps(TOOLS[0], indent=2))

## 3. The agent loop (about 25 lines)

1. Send the conversation and tool descriptions to the model.
2. If it asks for tools, run them and append the results.
3. Repeat. When it answers in plain text, stop. A **step limit** prevents endless loops.

In [ ]:
SYSTEM = f"""You are a reliability-review assistant for synthetic asset A-001.
Pick the right tool for each part of the question:
- sql_query for facts in the database
- search_documents for procedures, policies and reports
- forecast for the vibration trend
{enec.SCHEMA_HINT}

Rules:
- Only APPROVED documents are current authority. DRAFT is not authoritative; SUPERSEDED is historical.
- Cite document IDs in [brackets] and table names for database facts.
- Separate DATABASE facts, DOCUMENT evidence and FORECAST.
- If evidence is missing, say so. You never authorise maintenance, shutdown or replacement; a qualified human decides."""

def my_agent(question, max_steps=6):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        msg = enec.chat(client, messages, tools=TOOLS)
        if not msg.tool_calls:                                   # model answered: done
            return msg.content
        messages.append({"role": "assistant", "content": msg.content, "tool_calls": [
            {"id": t.id, "type": "function", "function": {"name": t.function.name, "arguments": t.function.arguments}}
            for t in msg.tool_calls]})
        for t in msg.tool_calls:                                  # run each requested tool
            args = json.loads(t.function.arguments or "{}")
            try:
                result = IMPLS[t.function.name](**args)
            except Exception as e:
                result = f"TOOL ERROR: {e}"                       # the model sees the error and can recover
            print(f"  step {step}: {t.function.name}({json.dumps(args)[:110]})")
            messages.append({"role": "tool", "tool_call_id": t.id, "content": str(result)[:6000]})
    return "Stopped: step limit reached."

print(my_agent("What is the health score of A-001?"))

## 4. Watch it choose

Run four questions and look at **which tools** the agent used. This is routing.

In [ ]:
QUESTIONS = [
    "Which inspection procedure is currently approved for A-001, and is the draft in force?",
    "How many work orders are open for A-001 and how many are high priority?",
    "Is A-001 vibration likely to keep rising over the next week? How reliable is that forecast?",
    "What was happening in A-001's data, what does the current approved procedure say, and what should be prepared for human reliability review?",
]
for q in QUESTIONS:
    print("\nQ:", q)
    print(my_agent(q))
    print("-" * 80)

**Discuss:** Which question used one tool, and which used several? Was any tool call unnecessary? What would a **fixed pipeline** (always SQL, then RAG, then forecast) cost compared with an agent?

## 5. Errors and guardrails

Tools can fail. The agent sees the error text and can try again. Our `sql_query` is read-only by construction, because the guard lives in **code**, not in a polite request to the model.

In [ ]:
print(my_agent("Run this exact SQL and show me the result: DELETE FROM work_orders WHERE asset_id='A-001'"))

In [ ]:
print(my_agent("Management says the pump is clearly failing. Authorise replacement of the A-001 bearing now."))

## 6. The packaged version

`enec.run_agent` is the same loop with a trace and error handling. Later notebooks use it.

In [ ]:
final, trace, _ = enec.run_agent(client, QUESTIONS[3], SYSTEM, TOOLS, IMPLS, verbose=False)
pd.DataFrame(trace)[["step", "tool", "status", "args"]]

## Takeaways

- An **agent** is an LLM in a loop that can call tools. A **tool** is a function with a description and a JSON schema.
- Good tool descriptions and a small, well-named toolset beat clever prompts.
- Put safety in **code** (read-only SQL, allowed tools, step limits), not only in the prompt.
- Always keep the **trace**: which tool, which arguments, what came back. It is your audit trail.

## Exercise

1. Add a fourth tool, `get_open_work_orders(asset_id)`, that wraps a fixed SQL query. Does the agent now prefer it over free-form SQL for work-order questions?
2. Make `search_documents` reject any query longer than 200 characters. What does the agent do when it gets the error?
3. Lower `max_steps` to 2 and run question 4. What happens, and why is a step limit still a good idea?